# *HL3 bypassed scenario for Man157 Network*

In [1]:
import sys
import os
# Navigate relative to the current working directory
sys.path.append(os.path.abspath('../../../src'))
import numpy as np
import pandas as pd
from sixdman.core.network import Network
from sixdman.core.band import Band, OpticalParameters
from sixdman.core.planning import PlanningTool
from sixdman.core.post_process import analyse_result
import json
from sixdman.utils.paths import get_project_root

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Use it to get paths
project_root = get_project_root()
results_dir = project_root / "results" / "MAN157_400G_CAGR_0.4" / "HL3Bypassed"
results_dir.mkdir(parents=True, exist_ok=True)

## *1. Create Network Instance*

In [3]:
# Initialize network
network = Network(topology_name = 'MAN157')

# Load topology from .mat file
network.load_topology(filepath = '../../../data/MAN157Nodes.mat', matrixName ='MAN157Nodes')

# Set hierarchical levels
hl_dict = network.define_hierarchy(
    HL1_standalone = [1, 5],
    HL2_standalone = [0, 2, 3, 4],
    HL4_standalone = list(range(6, 39)) + list(range(39, 157)), 
    HL4_colocated = [0, 2, 3, 4] + [1, 5]
)

## 2. *Define hierarchy node levels*

In [4]:
HL4_Standalone = hl_dict['HL4']['standalone']
HL4_colocated = hl_dict['HL4']['colocated']

HL2_Standalone = hl_dict['HL2']['standalone']
HL2_colocated = hl_dict['HL2']['colocated']

HL1_Standalone = hl_dict['HL1']['standalone']

arr = np.arange(network.adjacency_matrix.shape[0]) 
exclude_list = HL2_Standalone + HL4_Standalone + HL1_Standalone
Optical_Nodes = arr[~np.isin(arr, exclude_list)]


## 3. Define Transmission Bands

In [5]:
# Define C-band parameters
c_band_params = OpticalParameters(Rs_mat = 64e9)

# Create C-band instance
c_band = Band(
    name='C',
    start_freq = 190.65, # THz
    end_freq = 196.675, # THz
    opt_params = c_band_params,
    network_instance = network,
    channel_spacing = 0.075 # THz
    )

# Define L-band parameters
l_band_params = OpticalParameters(Rs_mat = 64e9)

# Create L-band instance
l_band = Band(
    name='L',
    start_freq = 184.525, # THz
    end_freq = 190.565, # THz
    opt_params = l_band_params,
    network_instance = network,
    channel_spacing = 0.075 # THz
)

In [6]:
# define C-band and L-band frequency slots
spectrum_C = c_band.calc_spectrum()
spectrum_L = l_band.calc_spectrum()

# concatenate C-band and KL-band to a sigle frequency spectrum
spectrum = np.concatenate((spectrum_C, spectrum_L))

f_c_axis = spectrum * 1e12  # Convert to Hz
Pch_dBm = np.arange(-6, -0.9, 0.1)  # Channel power in dBm
num_Ch_mat = np.arange(1, len(spectrum) - 1)  # Channel indices

# define total number of frequency slots
num_fslots = len(num_Ch_mat)

In [7]:
C_band_num_channels = int(4800 / 75)
supC_band_num_channels = int(1200 / 75)
L_band_num_channels = int(6000 / 75)

## 4. Create Planning Tool and Optimize Network

In [8]:
# Initialize planning tool
planner = PlanningTool(
    network_instance = network,
    bands = [c_band, l_band], 
    optical_parameters = c_band_params,
    grid_center = f_c_axis, 
    period_time = 10)

all_node_degree = network.get_node_degrees(np.arange(network.adjacency_matrix.shape[0]))[:, 1]


In [9]:
num_level_process = 4
minimum_hierarchy_level = 4
processing_level_list = [4, 2]

In [10]:
for hierarchy_level in processing_level_list:

    print(f"Processing hierarchy level: {hierarchy_level}")

    match hierarchy_level:

        case 2:
            HL_Standalone = HL2_Standalone
            HL_colocated = HL2_colocated
            HL_lower_Standalone = HL4_Standalone
            HL_all = np.concatenate((HL2_Standalone, HL2_colocated))
            capacity_updt_index = 1 # if no bypass scenario this variable is 0
            prev_hierarchy_level = 4

        case 4:
            HL_Standalone = HL4_Standalone
            HL_colocated = HL4_colocated
            HL_lower_Standalone = []
            HL_all = np.concatenate((HL4_Standalone, HL4_colocated))
            capacity_updt_index = 2
            prev_hierarchy_level = None

    _, subnetMatrix_HL = network.compute_hierarchy_subgraph(hierarchy_level, minimum_hierarchy_level)
    HL_connected_nodes = network.get_neighbor_nodes(HL_Standalone) - set(HL_lower_Standalone)
    
    file_name = results_dir /  f"{network.topology_name}_HL{hierarchy_level}_K_path_attributes.csv"

    if os.path.exists(file_name):

        print(f"Loading K-path attributes of HL{hierarchy_level} ...")

        K_path_attributes_df = pd.read_csv(file_name)
        K_path_attributes_df['links'] = K_path_attributes_df['links'].map(json.loads)
        K_path_attributes_df['nodes'] = K_path_attributes_df['nodes'].map(json.loads)

        # sort dataframes based on num_hops and distance (in order)
        K_path_attributes_df_sorted = K_path_attributes_df.groupby(['src_node'], group_keys = False).apply(lambda x: x.sort_values(['num_hops', 'distance']))

        # find disjoint pairs for standalone nodes
        pairs_disjoint = network.land_pair_finder(HL_Standalone, K_path_attributes_df_sorted, num_pairs = 1)

    else:

        print(f"Calculating K-path attributes of HL{hierarchy_level} ...")

        k_paths = 100
        source_not_found = HL_Standalone.copy()
        while len(source_not_found) != 0:

            # define a list to store path attributes for this iteration
            K_path_attributes = []

            # iterate through each "source_not_found" node only
            for src in HL_Standalone:
                for dest in HL_connected_nodes:
                    K_path_attributes = network.calculate_paths(subnetMatrix_HL, K_path_attributes, source = src, target = dest, k = k_paths)

            # Convert K_path_attributes list to a temporary DataFrame
            K_path_attributes_df = pd.DataFrame(K_path_attributes)

            # Optionally save to CSV (if you want to track progress)
            K_path_attributes_df.to_csv(file_name, index=False)

            # Sort by num_hops and distance
            K_path_attributes_df_sorted = K_path_attributes_df.groupby(['src_node'], group_keys=False).apply(
                lambda x: x.sort_values(['num_hops', 'distance'])
            )

            # find disjoint pairs from the sorted full dataset
            pairs_disjoint = network.land_pair_finder(HL_Standalone, K_path_attributes_df_sorted, num_pairs=1)

            # Update source_not_found to exclude newly matched source nodes
            source_not_found = np.setdiff1d(HL_Standalone, pairs_disjoint['src_node'].unique())

            print('Remaining src nodes: ', source_not_found)

            k_paths += 20
            print('k_new: ', k_paths)
    
    file_name = results_dir /  f"{network.topology_name}_HL{hierarchy_level}_K_path_attributes_colocated.csv"

    if os.path.exists(file_name):

        print(f"Loading K-path attributes of HL{hierarchy_level} colocated...")

        K_path_attributes_colocated_df = pd.read_csv(file_name)
        K_path_attributes_colocated_df['links'] = K_path_attributes_colocated_df['links'].map(json.loads)
        K_path_attributes_colocated_df['nodes'] = K_path_attributes_colocated_df['nodes'].map(json.loads)

        # sort dataframes based on num_hops and distance (in order)
        K_path_attributes_colocated_df_sorted = K_path_attributes_colocated_df.groupby(['src_node', 'dest_node'], group_keys = False).apply(lambda x: x.sort_values(['num_hops', 'distance']))

    else:

        print(f"Calculating K-path attributes of HL{hierarchy_level} colocated...")
        
        k_paths = 1

        # define a list to store path attributes
        K_path_attributes_colocated = []

        # iterate through each standalone HL4 node
        for src in HL_colocated:
            
            if (hierarchy_level == 3) and (src in HL2_Standalone):
                HL_connected_nodes_col = HL_connected_nodes - set(HL1_Standalone)
            else:
                HL_connected_nodes_col = HL_connected_nodes.copy()

            for dest in HL_connected_nodes:
                if src != dest:
                    K_path_attributes_colocated = network.calculate_paths(subnetMatrix_HL, K_path_attributes_colocated, source = src, target = dest, k = k_paths)

        # Convert K_path_attributes list to dataframe
        K_path_attributes_colocated_df = pd.DataFrame(K_path_attributes_colocated)

        # save dataframe to csv file
        K_path_attributes_colocated_df.to_csv(file_name, index = False)

        # sort dataframes based on num_hops and distance (in order)
        K_path_attributes_colocated_df_sorted = K_path_attributes_colocated_df.groupby(['src_node', 'dest_node'], group_keys = False).apply(lambda x: x.sort_values(['num_hops', 'distance']))

    print(f"process link GSMR of HL{hierarchy_level} ...")

    GSNR_opt_link, _, _, power_opt_link = c_band.process_link_gsnr(f_c_axis = f_c_axis, 
                                                                    Pch_dBm = Pch_dBm, 
                                                                    num_Ch_mat = num_Ch_mat,
                                                                    spectrum_C = spectrum_C,
                                                                    Nspan_array = np.ones(network.all_links.shape[0], dtype=int),
                                                                    hierarchy_level = hierarchy_level, 
                                                                    minimum_hierarchy_level = minimum_hierarchy_level, 
                                                                    result_directory = results_dir)
    
    planner.initialize_planner(num_fslots = num_fslots, 
                               hierarchy_level = hierarchy_level,
                               minimum_hierarchy_level = minimum_hierarchy_level, 
                               SR = 64 * 1e9, # Symbol rate of 75G channels (in Hz)
                               Max_bit_rate_BVT = np.array([400, 320, 260, 200, 120, 64]), # 400 Gbps BVTs
                               Ref_license_capacity = np.array([100, 80, 65, 50, 30, 16]), # reference license capacities corresponding to BVTs
                               FP_max_num = 20, 
                               band_sepration_idx = [C_band_num_channels, 
                                                     C_band_num_channels + supC_band_num_channels]
                               )
    
    if hierarchy_level == minimum_hierarchy_level:

        # generate port capacity for HL4 nodes uisng Monte Carlo simulation
        planner.generate_initial_traffic_profile(num_nodes = len(HL_all),
                                                 optical_nodes = Optical_Nodes,
                                                 monteCarlo_steps = 100,
                                                 min_rate = 20, # Gbps
                                                 max_rate = 200, # Gbps
                                                 seed = 50, 
                                                 result_directory = results_dir)

        # Traffic growth simulation over 10 years
        planner.simulate_traffic_annual(lowest_hierarchy_dict = hl_dict[f"HL{hierarchy_level}"], 
                                        CAGR = 0.4,
                                        result_directory = results_dir)
        
    print(f"running planner for HL{hierarchy_level} ...")

    # run the planner for the current hierarchy level    
    planner.run_planner(hierarchy_level = hierarchy_level,
                        prev_hierarchy_level = prev_hierarchy_level, 
                        pairs_disjoint = pairs_disjoint,
                        kpair_standalone = 1,
                        kpair_colocated = 1,
                        candidate_paths_standalone_df = K_path_attributes_df,
                        candidate_paths_colocated_df = K_path_attributes_colocated_df_sorted,
                        P_opt_links = power_opt_link,
                        GSNR_opt_link = GSNR_opt_link,
                        Nspan_array = np.ones(len(network.all_links)),
                        all_node_degree = all_node_degree,
                        minimum_level = minimum_hierarchy_level,
                        node_cap_update_idx = capacity_updt_index, 
                        result_directory = results_dir)


Processing hierarchy level: 4
Loading K-path attributes of HL4 ...
Loading K-path attributes of HL4 colocated...
process link GSMR of HL4 ...
Loading precomputed link GSNR analysis
Loading precomputed HL_capacity_final ...
Loading precomputed Traffic Matrix ...
running planner for HL4 ...
Processing Year:  1
Processing standalone node 6
Processing standalone node 7
Processing standalone node 8
Processing standalone node 9
Processing standalone node 10
Processing standalone node 11
Processing standalone node 12
Processing standalone node 13
Processing standalone node 14
Processing standalone node 15
Processing standalone node 16
Processing standalone node 17
Processing standalone node 18
Processing standalone node 19
Processing standalone node 20
Processing standalone node 21
Processing standalone node 22
Processing standalone node 23
Processing standalone node 24
Processing standalone node 25
Processing standalone node 26
Processing standalone node 27
Processing standalone node 28
Proc

In [11]:
analysing = analyse_result(network, 10, processing_level_list, results_dir, save_directory = results_dir)
analysing._load_data()
year = np.arange(1, 11)

In [12]:
cost_df_C_IRU_0v5 = analysing.calc_cost(save_flag = 1, 
                    save_suffix = "_HL3Bypassed_C_IRU_0v5", 
                    C_IRU = 0.5,)

cost_df_C_IRU_0v01 = analysing.calc_cost(save_flag = 1, 
                    save_suffix = "_HL3Bypassed_C_IRU_0v01", 
                    C_IRU = 0.01,)

cost_df_C_IRU_5 = analysing.calc_cost(save_flag = 1, 
                    save_suffix = "_HL3Bypassed_C_IRU_5", 
                    C_IRU = 5,)

In [13]:
cost_df_C_IRU_0v5

,OPEX,Capex_RoB,Capex_MCS,Capex_100GL,CAPEX
0,2450.22,836.000000,36.050000,1118.372000,1990.422000
1,2450.22,0.000000,2.625000,238.488000,241.113000
2,2450.22,0.000000,4.375000,329.104000,333.479000
3,2450.22,10.471356,15.098894,611.868622,637.438872
4,2450.22,19.745986,15.427846,768.924778,804.098610
5,2486.36,29.245395,26.182877,1150.899963,1206.328235
6,2523.79,15.613933,33.860710,1568.348853,1617.823496
7,2668.14,29.141807,48.768520,2236.423263,2314.333591
8,2860.98,34.963813,67.945825,3086.761381,3189.671019
9,3239.30,63.842261,94.093649,4307.691606,4465.627516


In [14]:
node_capacity_profile_array_HL3Bypassed = np.load(results_dir / "MAN157_HL2_node_capacity_profile_array.npz")['node_capacity_profile_array']

In [15]:
node_capacity_profile_array_HL3Bypassed_HL4 = node_capacity_profile_array_HL3Bypassed[:, :, 3].cumsum(axis = 0)
node_capacity_profile_array_HL3Bypassed_HL2 = node_capacity_profile_array_HL3Bypassed[:, :, 2].cumsum(axis = 0)
node_capacity_profile_array_HL3Bypassed_HL1 = node_capacity_profile_array_HL3Bypassed[:, :, 1].cumsum(axis = 0)

ref_routers_cap = np.array([400, 800, 1600, 3200, 6400, 12800, 25600])
ref_routers_cost_metric = np.array([12, 24, 48, 96, 192, 384, 768]) # cost units in the comprehensive paper
ref_routers_cost_metric = ref_routers_cost_metric * 2000 / 15000 # cost unit 0f paper is 2000, our work is 15000
ref_routers_power_consumption = np.array([2000, 2500, 3100, 3900, 4900, 6100, 7600]) # in Watts
ref_year = 2 # year five is the reference for selection of router

In [16]:
# HL3Bypassed Scenario
HL4_SA_num_routers_HL3Bypassed = []
HL4_SA_selected_router_HL3Bypassed = []

HL4_co_num_routers_HL3Bypassed = []
HL4_co_selected_router_HL3Bypassed = []

HL2_SA_num_routers_HL3Bypassed = []
HL2_SA_selected_router_HL3Bypassed = []

HL2_co_num_routers_HL3Bypassed = []
HL2_co_selected_router_HL3Bypassed = []

HL1_SA_num_routers_HL3Bypassed = []
HL1_SA_selected_router_HL3Bypassed = []

routers_tracker_array_HL3Bypassed = np.zeros(shape = (len(ref_routers_cap), len(year)), dtype = np.int64)

for hl4_sa in hl_dict['HL4']['standalone']:
    ref_traffic = node_capacity_profile_array_HL3Bypassed_HL4[ref_year - 1, hl4_sa]
    idx = np.searchsorted(ref_routers_cap, ref_traffic, side='left')
    closest_bigger_router_cap = ref_routers_cap[idx] if idx < len(ref_routers_cap) else ref_routers_cap[-1]
    HL4_SA_selected_router_HL3Bypassed.append(closest_bigger_router_cap)
    idx = idx if idx < len(ref_routers_cap) else len(ref_routers_cap) - 1
    routers_tracker_array_HL3Bypassed[idx, :] += (node_capacity_profile_array_HL3Bypassed_HL4[:, hl4_sa] // closest_bigger_router_cap).astype(int) + 1
    HL4_SA_num_routers_HL3Bypassed.append(node_capacity_profile_array_HL3Bypassed_HL4[:, hl4_sa] // closest_bigger_router_cap + 1)

for hl4_co in hl_dict['HL4']['colocated']:
    ref_traffic = node_capacity_profile_array_HL3Bypassed_HL4[ref_year - 1, hl4_co]
    idx = np.searchsorted(ref_routers_cap, ref_traffic, side='left')
    closest_bigger_router_cap = ref_routers_cap[idx] if idx < len(ref_routers_cap) else ref_routers_cap[-1]
    HL4_co_selected_router_HL3Bypassed.append(closest_bigger_router_cap)
    idx = idx if idx < len(ref_routers_cap) else len(ref_routers_cap) - 1
    routers_tracker_array_HL3Bypassed[idx, :] += (node_capacity_profile_array_HL3Bypassed_HL4[:, hl4_co] // closest_bigger_router_cap).astype(int) + 1
    HL4_co_num_routers_HL3Bypassed.append(node_capacity_profile_array_HL3Bypassed_HL4[:, hl4_co] // closest_bigger_router_cap + 1)

for hl2_sa in hl_dict['HL2']['standalone']:
    ref_traffic = node_capacity_profile_array_HL3Bypassed_HL2[ref_year - 1, hl2_sa]
    idx = np.searchsorted(ref_routers_cap, ref_traffic, side='left')
    closest_bigger_router_cap = ref_routers_cap[idx] if idx < len(ref_routers_cap) else ref_routers_cap[-1]
    HL2_SA_selected_router_HL3Bypassed.append(closest_bigger_router_cap)
    idx = idx if idx < len(ref_routers_cap) else len(ref_routers_cap) - 1
    routers_tracker_array_HL3Bypassed[idx, :] += (node_capacity_profile_array_HL3Bypassed_HL2[:, hl2_sa] // closest_bigger_router_cap).astype(int) + 1
    HL2_SA_num_routers_HL3Bypassed.append(node_capacity_profile_array_HL3Bypassed_HL2[:, hl2_sa] // closest_bigger_router_cap + 1)

for hl2_co in hl_dict['HL2']['colocated']:
    ref_traffic = node_capacity_profile_array_HL3Bypassed_HL2[ref_year - 1, hl2_co]
    idx = np.searchsorted(ref_routers_cap, ref_traffic, side='left')
    closest_bigger_router_cap = ref_routers_cap[idx] if idx < len(ref_routers_cap) else ref_routers_cap[-1]
    HL2_co_selected_router_HL3Bypassed.append(closest_bigger_router_cap)
    idx = idx if idx < len(ref_routers_cap) else len(ref_routers_cap) - 1
    routers_tracker_array_HL3Bypassed[idx, :] += (node_capacity_profile_array_HL3Bypassed_HL2[:, hl2_co] // closest_bigger_router_cap).astype(int) + 1
    HL2_co_num_routers_HL3Bypassed.append(node_capacity_profile_array_HL3Bypassed_HL2[:, hl2_co] // closest_bigger_router_cap + 1)

for hl1_sa in hl_dict['HL1']['standalone']:
    ref_traffic = node_capacity_profile_array_HL3Bypassed_HL1[ref_year - 1, hl1_sa]
    idx = np.searchsorted(ref_routers_cap, ref_traffic, side='left')
    closest_bigger_router_cap = ref_routers_cap[idx] if idx < len(ref_routers_cap) else ref_routers_cap[-1]
    HL1_SA_selected_router_HL3Bypassed.append(closest_bigger_router_cap)
    idx = idx if idx < len(ref_routers_cap) else len(ref_routers_cap) - 1
    routers_tracker_array_HL3Bypassed[idx, :] += (node_capacity_profile_array_HL3Bypassed_HL1[:, hl1_sa] // closest_bigger_router_cap).astype(int) + 1
    HL1_SA_num_routers_HL3Bypassed.append(node_capacity_profile_array_HL3Bypassed_HL1[:, hl1_sa] // closest_bigger_router_cap + 1)
    
num_routers_HL4_SA_total_yearly_HL3Bypassed = np.sum(HL4_SA_num_routers_HL3Bypassed, axis = 0)
num_routers_HL4_co_total_yearly_HL3Bypassed = np.sum(HL4_co_num_routers_HL3Bypassed, axis = 0)
num_routers_HL2_SA_total_yearly_HL3Bypassed = np.sum(HL2_SA_num_routers_HL3Bypassed, axis = 0)
num_routers_HL2_co_total_yearly_HL3Bypassed = np.sum(HL2_co_num_routers_HL3Bypassed, axis = 0)
num_routers_HL1_SA_total_yearly_HL3Bypassed = np.sum(HL1_SA_num_routers_HL3Bypassed, axis = 0)

total_num_router_yearly_HL3Bypassed = num_routers_HL4_SA_total_yearly_HL3Bypassed + \
    num_routers_HL4_co_total_yearly_HL3Bypassed + num_routers_HL2_SA_total_yearly_HL3Bypassed \
        + num_routers_HL2_co_total_yearly_HL3Bypassed + num_routers_HL1_SA_total_yearly_HL3Bypassed

In [17]:
# np.savez(results_dir / "MAN157_routers_tracker_array_HL3Bypassed.npz", routers_tracker_array = routers_tracker_array_HL3Bypassed)

In [18]:
# HL3Bypassed Scenario
HL4_SA_cost_HL3Bypassed = []
HL4_co_cost_HL3Bypassed = []
HL2_SA_cost_HL3Bypassed = []
HL2_co_cost_HL3Bypassed = []
HL1_SA_cost_HL3Bypassed = []

HL4_SA_power_HL3Bypassed = []
HL4_co_power_HL3Bypassed = []
HL2_SA_power_HL3Bypassed = []
HL2_co_power_HL3Bypassed = []
HL1_SA_power_HL3Bypassed = []

indices = [np.where(ref_routers_cap == v)[0][0] for v in HL4_SA_selected_router_HL3Bypassed]
HL4_SA_HL3Bypassed_cost_metrics = ref_routers_cost_metric[indices]
HL4_SA_HL3Bypassed_power_metrics = ref_routers_power_consumption[indices]
HL4_SA_cost_HL3Bypassed = [ai * bi for ai, bi in zip(HL4_SA_num_routers_HL3Bypassed, HL4_SA_HL3Bypassed_cost_metrics)]
HL4_SA_power_HL3Bypassed = [ai * bi for ai, bi in zip(HL4_SA_num_routers_HL3Bypassed, HL4_SA_HL3Bypassed_power_metrics)]

indices = [np.where(ref_routers_cap == v)[0][0] for v in HL4_co_selected_router_HL3Bypassed]
HL4_co_HL3Bypassed_cost_metrics = ref_routers_cost_metric[indices]
HL4_co_HL3Bypassed_power_metrics = ref_routers_power_consumption[indices]
HL4_co_cost_HL3Bypassed = [ai * bi for ai, bi in zip(HL4_co_num_routers_HL3Bypassed, HL4_co_HL3Bypassed_cost_metrics)]
HL4_co_power_HL3Bypassed = [ai * bi for ai, bi in zip(HL4_co_num_routers_HL3Bypassed, HL4_co_HL3Bypassed_power_metrics)]

indices = [np.where(ref_routers_cap == v)[0][0] for v in HL2_SA_selected_router_HL3Bypassed]
HL2_SA_HL3Bypassed_cost_metrics = ref_routers_cost_metric[indices]
HL2_SA_HL3Bypassed_power_metrics = ref_routers_power_consumption[indices]
HL2_SA_cost_HL3Bypassed = [ai * bi for ai, bi in zip(HL2_SA_num_routers_HL3Bypassed, HL2_SA_HL3Bypassed_cost_metrics)]
HL2_SA_power_HL3Bypassed = [ai * bi for ai, bi in zip(HL2_SA_num_routers_HL3Bypassed, HL2_SA_HL3Bypassed_power_metrics)]

indices = [np.where(ref_routers_cap == v)[0][0] for v in HL2_co_selected_router_HL3Bypassed]
HL2_co_HL3Bypassed_cost_metrics = ref_routers_cost_metric[indices]
HL2_co_HL3Bypassed_power_metrics = ref_routers_power_consumption[indices]
HL2_co_cost_HL3Bypassed = [ai * bi for ai, bi in zip(HL2_co_num_routers_HL3Bypassed, HL2_co_HL3Bypassed_cost_metrics)]
HL2_co_power_HL3Bypassed = [ai * bi for ai, bi in zip(HL2_co_num_routers_HL3Bypassed, HL2_co_HL3Bypassed_power_metrics)]

indices = [np.where(ref_routers_cap == v)[0][0] for v in HL1_SA_selected_router_HL3Bypassed]
HL1_SA_HL3Bypassed_cost_metrics = ref_routers_cost_metric[indices]
HL1_SA_HL3Bypassed_power_metrics = ref_routers_power_consumption[indices]
HL1_SA_cost_HL3Bypassed = [ai * bi for ai, bi in zip(HL1_SA_num_routers_HL3Bypassed, HL1_SA_HL3Bypassed_cost_metrics)]
HL1_SA_power_HL3Bypassed = [ai * bi for ai, bi in zip(HL1_SA_num_routers_HL3Bypassed, HL2_SA_HL3Bypassed_power_metrics)]

IP_total_cost_HL3Bypassed = np.sum(HL4_SA_cost_HL3Bypassed, axis = 0) + np.sum(HL4_co_cost_HL3Bypassed, axis = 0) + \
    np.sum(HL2_SA_cost_HL3Bypassed, axis = 0) \
        + np.sum(HL2_co_cost_HL3Bypassed, axis = 0) + np.sum(HL1_SA_cost_HL3Bypassed, axis = 0)
IP_total_power_HL3Bypassed = np.sum(HL4_SA_power_HL3Bypassed, axis = 0) + np.sum(HL4_co_power_HL3Bypassed, axis = 0) + \
    np.sum(HL2_SA_power_HL3Bypassed, axis = 0) \
        + np.sum(HL2_co_power_HL3Bypassed, axis = 0) + np.sum(HL1_SA_power_HL3Bypassed, axis = 0)

In [19]:
OTCO_annual = (cost_df_C_IRU_0v5['OPEX'] + cost_df_C_IRU_0v5['CAPEX']).values
TCO_Final_year = (IP_total_cost_HL3Bypassed[-1] + OTCO_annual.sum())
TCO_Final_year

np.float64(47458.80533931986)

In [20]:
OTCO_annual = (cost_df_C_IRU_0v01['OPEX'] + cost_df_C_IRU_0v01['CAPEX']).values
TCO_Final_year = (IP_total_cost_HL3Bypassed[-1] + OTCO_annual.sum())
TCO_Final_year

np.float64(21949.728739319857)

In [21]:
OTCO_annual = (cost_df_C_IRU_5['OPEX'] + cost_df_C_IRU_5['CAPEX']).values
TCO_Final_year = (IP_total_cost_HL3Bypassed[-1] + OTCO_annual.sum())
TCO_Final_year

np.float64(281725.8353393199)

In [ ]:
# np.savez(results_dir / "MAN157_IP_Cost_Power_HL3Bypassed_RefYear_2.npz", 
#          IP_Cost_Cum = IP_total_cost_HL3Bypassed, 
#          IP_Power_Cum = IP_total_power_HL3Bypassed 
#          )

In [23]:
power_unit_ROADM = 72 #910 # Watt
power_unit_MCS = 10 # Watt
typ_power_400G_BVT = 160 # Watt

In [24]:
Total_BVT_HL3bypassed = analysing.bvt_data['HL4']['HL_BVTNum_All'].sum(axis = 1) +\
        analysing.bvt_data['HL2']['HL_BVTNum_All'].sum(axis = 1)

In [25]:
degree_links_cum = analysing.link_data['HL2']['HL_CDegree_Domain'].sum(axis = 1) +\
    analysing.link_data['HL2']['HL_SuperCDegree_Domain'].sum(axis = 1) +\
        analysing.link_data['HL2']['HL_LDegree_Domain'].sum(axis = 1)

In [26]:
# Energy Consumption Between scenarios
energy_consumption_annual_HL3Bypassed = (np.diff(Total_BVT_HL3bypassed, prepend=0) * typ_power_400G_BVT +\
    np.diff(degree_links_cum, prepend=0) * power_unit_ROADM +\
        np.diff(Total_BVT_HL3bypassed, prepend=0) / 16 * power_unit_MCS) * 365 * 24 / 1e6

total_served_traffic_HL3Bypassed = node_capacity_profile_array_HL3Bypassed_HL4.sum(axis = 1)
IP_energy_consumption_annual_HL3Bypassed = np.diff(IP_total_power_HL3Bypassed * 365 * 24 / 1e6, prepend=0)

# np.savez(results_dir / 'MAN157_energy_optical_electrical_HL3Bypassed.npz',
#          optical_energy_annual = energy_consumption_annual_HL3Bypassed,
#          IP_energy_annual = IP_energy_consumption_annual_HL3Bypassed, 
#          total_served_traffic = total_served_traffic_HL3Bypassed)

ENERGY_Final_Year = (energy_consumption_annual_HL3Bypassed + IP_energy_consumption_annual_HL3Bypassed).cumsum()[-1]
ENERGY_Final_Year


np.float64(32559.0366)

In [27]:
latency_core_array_SA_global = [
    np.load(results_dir / 'MAN157_HL4_segments_latency.npz', allow_pickle = True)['latency_std'], 
    np.load(results_dir / 'MAN157_HL2_segments_latency.npz', allow_pickle = True)['latency_std']
]

destination_core_array_SA_global = [
    np.load(results_dir / 'MAN157_HL4_segments_latency.npz', allow_pickle = True)['destinations_std'], 
    np.load(results_dir / 'MAN157_HL2_segments_latency.npz', allow_pickle = True)['destinations_std']
]

latency_core_array_co_global = [
    np.load(results_dir / 'MAN157_HL4_segments_latency.npz', allow_pickle = True)['latency_co'], 
    np.load(results_dir / 'MAN157_HL2_segments_latency.npz', allow_pickle = True)['latency_co']
]

destination_core_array_c0_global = [
    np.load(results_dir / 'MAN157_HL4_segments_latency.npz', allow_pickle = True)['destinations_co'], 
    np.load(results_dir / 'MAN157_HL2_segments_latency.npz', allow_pickle = True)['destinations_co']
]

In [28]:
latency_annual = []
for year in range(1, planner.period_time + 1):
    latency = analysing.calc_E2E_latency_Total(latency_core_array_SA_global = latency_core_array_SA_global, 
                                               destination_core_array_SA_global = destination_core_array_SA_global, 
                                               latency_core_array_co_global = latency_core_array_co_global, 
                                               destination_core_array_co_global = destination_core_array_c0_global, 
                                               processing_level_list = processing_level_list, 
                                               node_capacity_profile_annual = node_capacity_profile_array_HL3Bypassed.cumsum(axis = 0), 
                                               node_cap_index_list = [2, 1],           
                                               reference_year = 5, 
                                               reference_router_capacities = ref_routers_cap, 
                                               year = year,  
                                               save_flag = 0)
    latency_annual.append(latency[latency != 0])
    
latency_annual = np.array(latency_annual)
avg_latency = np.mean(latency_annual, axis = 0)

In [29]:
distance = analysing.calc_E2E_distance_Total(latency_core_array_SA_global = latency_core_array_SA_global, 
                                            destination_core_array_SA_global = destination_core_array_SA_global, 
                                            latency_core_array_co_global = latency_core_array_co_global, 
                                            destination_core_array_co_global = destination_core_array_c0_global, 
                                            processing_level_list = processing_level_list,  
                                            save_flag = 0)
distance = distance[distance != 0]

In [30]:
# np.savez(results_dir / 'MAN157_Average_Latency.npz', 
#          latency = avg_latency, 
#          distance = distance)